# 01 — Exploration

Reads the local DuckDB built by `python run.py` and reproduces the Page-1 story: volume by type, value by type, fraud rate by type, amount distribution, time trend.
Balances are never used as features (publisher leakage warning).

In [ ]:
import duckdb
from pathlib import Path

con = duckdb.connect(str(Path('..') / 'data' / 'processed' / 'fraudops.duckdb'))
print(con.execute('SELECT COUNT(*), SUM(is_fraud), 100.0*SUM(is_fraud)/COUNT(*) FROM fact_transactions').fetchdf().to_string(index=False, header=False))

In [ ]:
by_type = con.execute('SELECT transaction_type, COUNT(*) n, SUM(amount) v, SUM(is_fraud) f, 100.0*SUM(is_fraud)/COUNT(*) r FROM fact_transactions GROUP BY transaction_type ORDER BY r DESC').fetchdf()
by_type

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns, numpy as np
full = con.execute('SELECT transaction_type, amount FROM fact_transactions').fetchdf()
plt.figure(figsize=(7, 4)); sns.countplot(data=full, x='transaction_type'); plt.title('Count by type'); plt.show()
plt.figure(figsize=(7, 4)); plt.hist(np.log1p(full['amount']), bins=80); plt.title('Amount log1p'); plt.show()